In [1]:
##modules
#%matplotlib widget
%matplotlib inline
#
#%matplotlib qt
import mne
import numpy as np
import matplotlib
# Establecer un backend interactivo, como 'Qt5Agg', 'GTK3Agg', etc.
# Esto depende de los backends disponibles en tu sistema.

import matplotlib.pyplot as plt

#matplotlib.use('TkAgg')  # Asegúrate de que este backend está instalado.

import pandas as pd 
import os
import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from os.path import join as pathjoin
from time import time

from pathlib import Path

from autoreject import AutoReject

# aplicar la acf EN epochs

from statsmodels.tsa.stattools import acf


from scipy.stats import permutation_test
sys.path.append(r"G:\apuntes_mne\code_MOUS\scripts_functions_created")

# Ahora importa la función
from print5 import print5


import pickle

from scipy.stats import gaussian_kde
import numpy as np
import matplotlib.pyplot as plt

# Import required code for visualizing example models
from fooof import FOOOF
from fooof.sim.gen import gen_power_spectrum
from fooof.sim.utils import set_random_seed
from fooof.plts.spectra import plot_spectra
from fooof.plts.annotate import plot_annotated_model
from fooof import FOOOFGroup,Bands
from fooof.analysis.periodic import get_band_peak_group, get_band_peak


C:\Users\UCM\AppData\Local\Temp\ipykernel_20152\3946965986.py:48: DeprecationWarning: 
The `fooof` package is being deprecated and replaced by the `specparam` (spectral parameterization) package.
This version of `fooof` (1.1) is fully functional, but will not be further updated.
New projects are recommended to update to using `specparam` (see Changelog for details).
  from fooof import FOOOF


In [2]:
# Paths
try:
    # Si se ejecuta como SCRIPT .py: usar __file__
    sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
except NameError:
    # Si se ejecuta como NOTEBOOK Jupyter: usar path relativo
    sys.path.append("..")  # sube un nivel desde la carpeta actual del notebook


# --- Configuración dinámica de rutas ---
from get_paths_MOUS import get_paths_MOUS

# Parámetros editables
disco = "g"
modality = "visual"
layer_script = "event"
subj = "sub-V1001"
#subj = sys.argv[1] ## name of participant list


# Generar variables automáticamente
path_dict = get_paths_MOUS(disco=disco, modality=modality, layer_script=layer_script, subj=subj)
globals().update(path_dict)

# Mostrar todos los paths generados
print("\n📁 Rutas generadas:")
for k, v in path_dict.items():
    print(f"{k:<20} → {v}")
    
        
    
with open(datadir / f"subjects_remove_{modality}.pkl", "rb") as f:
    subjects_remove = pickle.load(f)
    



📁 Rutas generadas:
datadir              → g:\MOUS_204\MOUS_visual
general_datadir      → g:\MOUS_204
output_preproc       → g:\MOUS_204\MOUS_visual\output_preproc
mri_dir              → g:\MOUS_204\sub-V1001\anat
meg_dir              → g:\MOUS_204\sub-V1001\meg
preproc_path         → g:\MOUS_204\MOUS_visual\output_preproc\preproc_event
channels_structure_path → g:\MOUS_204\channels_structure
epochs_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\epochs_event
ICA_path             → g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\ICA_event
epochs_clean_path    → g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\epochs_clean_event
evoked_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\evoked_event
source_path          → g:\MOUS_204\MOUS_visual\output_source\source_event
raw_hsp_path         → g:\MOUS_204\MOUS_visual\output_source\source_event\raw_hsp
fwd_path             → g:\MOUS_204\MOUS_visual\output_source\source_event\fwd
inverse_pat

In [3]:
conditions=["zinnen", "woorden"]
# -----------------------------------------------------
# get data from epochs
# -----------------------------------------------------
#take into account that data shape is (n_epochs, n_channels, n_times), but it does not differenciate between conditions
# epochs_clean = Path("ruta/a/tu/carpeta")

subjects = []

for archivo in epochs_clean_path.iterdir():
    # Detecta modalidad
    if modality == "visual":
        subject_prefix = 'sub-V1'
    elif modality == "auditory":
        subject_prefix = 'sub-A2'

    # Filtra solo archivos .fif que comiencen con el prefijo de sujeto
    if archivo.is_file() and archivo.suffix == ".fif" and archivo.name.startswith(subject_prefix):
        # Extrae solo el ID del sujeto (todo hasta el primer "_")
        subject_id = archivo.name.split("_")[0]

        subjects.append(subject_id)

print(subjects)



['sub-V1001', 'sub-V1002', 'sub-V1003', 'sub-V1004', 'sub-V1005', 'sub-V1007', 'sub-V1008', 'sub-V1009', 'sub-V1010', 'sub-V1011', 'sub-V1012', 'sub-V1013', 'sub-V1015', 'sub-V1016', 'sub-V1019', 'sub-V1020', 'sub-V1022', 'sub-V1024', 'sub-V1025', 'sub-V1026', 'sub-V1027', 'sub-V1028', 'sub-V1029', 'sub-V1030', 'sub-V1031', 'sub-V1032', 'sub-V1033', 'sub-V1034', 'sub-V1035', 'sub-V1036', 'sub-V1037', 'sub-V1038', 'sub-V1039', 'sub-V1040', 'sub-V1042', 'sub-V1044', 'sub-V1045', 'sub-V1046', 'sub-V1048', 'sub-V1049', 'sub-V1050', 'sub-V1052', 'sub-V1053', 'sub-V1054', 'sub-V1055', 'sub-V1057', 'sub-V1058', 'sub-V1059', 'sub-V1061', 'sub-V1062', 'sub-V1063', 'sub-V1064', 'sub-V1065', 'sub-V1066', 'sub-V1068', 'sub-V1069', 'sub-V1070', 'sub-V1071', 'sub-V1072', 'sub-V1073', 'sub-V1074', 'sub-V1075', 'sub-V1076', 'sub-V1077', 'sub-V1078', 'sub-V1079', 'sub-V1080', 'sub-V1081', 'sub-V1083', 'sub-V1084', 'sub-V1085', 'sub-V1086', 'sub-V1087', 'sub-V1088', 'sub-V1089', 'sub-V1090', 'sub-V1092'

In [4]:
def check_nans(data, nan_policy='zero'):
    """Check an array for nan values, and replace, based on policy."""

    # Find where there are nan values in the data
    nan_inds = np.where(np.isnan(data))

    # Apply desired nan policy to data
    if nan_policy == 'zero':
        data[nan_inds] = 0
    elif nan_policy == 'mean':
        data[nan_inds] = np.nanmean(data)
    else:
        raise ValueError('Nan policy not understood.')

    return data

In [5]:
dict_isc= pd.read_pickle(ISC_block_path /f"ISC_results_block.pkl")
dict_woorden_block=dict_isc['dict_isc_WOORDEN']
dict_zinnen_block = dict_isc['dict_isc_ZINNEN']
##channels significant adjusted in each condition
numbers_channels_woorden= dict_woorden_block["significant_channels_adjusted"]
numbers_channels_zinnen= dict_zinnen_block["significant_channels_adjusted"]
print("numbers_channels_woorden", numbers_channels_woorden, "numbers_channels_zinnen", numbers_channels_zinnen)
print("len(numbers_channels_woorden)",len(numbers_channels_woorden),  "len(numbers_channels_zinnen)",len(numbers_channels_zinnen))

names_channels_woorden=dict_woorden_block["significant_channels_adjusted_names"]
names_channels_zinnen=dict_zinnen_block["significant_channels_adjusted_names"]

print("names_channels_woorden", names_channels_woorden)
print("names_channels_zinnen", names_channels_zinnen)

h_subj =0
path_epochs = epochs_clean_path / f"{subjects[h_subj]}_epochs_{layer_script}-epo.fif"
epochs = mne.read_epochs(path_epochs, preload=False)
#establecimiento de canales palabras, canales frases y canales mixtos
# Tomamos el orden original de los canales del objeto epochs
all_channels = epochs.ch_names
all_channels_numbers= [epochs.ch_names.index(ch) for ch in epochs.ch_names]


del epochs

# Palabras
numbers_channels_only_woorden = [
    ch for ch in all_channels_numbers if ch in numbers_channels_woorden and ch not in numbers_channels_zinnen
]

numbers_channels_only_zinnen = [
    ch for ch in all_channels_numbers if ch in numbers_channels_zinnen and ch not in numbers_channels_woorden
]

numbers_channels_intersection = [
    ch for ch in all_channels_numbers if ch in numbers_channels_woorden and ch in numbers_channels_zinnen
]

# Lo mismo pero usando nombres
names_channels_only_woorden = [
    ch for ch in all_channels if ch in names_channels_woorden and ch not in names_channels_zinnen
]

names_channels_only_zinnen = [
    ch for ch in all_channels if ch in names_channels_zinnen and ch not in names_channels_woorden
]

names_channels_intersection = [
    ch for ch in all_channels if ch in names_channels_woorden and ch in names_channels_zinnen
]
# ---------------------------
# Print resumen
# ---------------------------

print(f"len(significant_channels_only_woorden): {len(numbers_channels_only_woorden)}, "
      f"len(significant_channels_only_zinnen): {len(numbers_channels_only_zinnen)}, "
      f"len(significant_channels_intersection): {len(numbers_channels_intersection)}")



print("\n✅ Only woorden (names):", names_channels_only_woorden)
print("✅ Only zinnen (names):", names_channels_only_zinnen)
print("✅ Intersection (names):", names_channels_intersection)


dict_select_channels={
    "names_channels_only_woorden": names_channels_only_woorden,
    "names_channels_only_zinnen": names_channels_only_zinnen,
    "names_channels_intersection": names_channels_intersection
}

numbers_channels_woorden [  2   3   4   5   7   8   9  10  11  12  13  14  17  18  35  36  40  41
  42  43  45  46  47  48  49  51  52  53  54  56  57  58  60  61  62  64
  65  66  68  69  70  73  75  77  78  80  81  82  83  84  85  86  87  88
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126
 127 128 131 132 133 134 135 136 137 138 139 140 141 142 143 144 146 147
 148 149 150 151 182 183 184 188 192 196 199 200 201 202 204 206 207 209
 210 211 212 214 215 216 217 221 222 223 224 225 226 228 229 230 231 232
 233 234 235 236 237 238 239 240 241 242 243 244 245 246 247 248 249 250
 251 252 253 254 255 256 257 258] numbers_channels_zinnen [  2   3   4   5   7   8   9  10  11  12  13  14  17  18  19  24  25  26
  29  30  31  33  34  35  36  37  38  39  40  41  42  43  44  45  46  47
  48  49  51  52  53  54  55  56  57  58  59  62  66  75  77  78  79  80
  81  82  83  84  85  86  87  88  89  90 

In [6]:


def compute_fooof_subject(subject,epochs, bands,dict_select_channels,change_name=None ,aperiodic_mode="fixed", select_highest=False, select_channels=None,return_fg_subject=False ):

    #list to append the data frames 
    list_df = []
    #dictionary to append the fooof groups
    dict_fg={}
    
    
    #now 
    condition_names = list(epochs.event_id.keys())

    
    #but i will keep the original names to select the epochs
    for i_cond in range(0,len(condition_names)):
        
        #now i take the original name to select the epochs
        cond=condition_names[i_cond]
    
        #now i select the epochs for that condition
        epochs_cond = epochs[cond]
        
        #if any condition has no epochs i skip it
        if len(epochs_cond) == 0:
            continue
    
        #if change name i chanege the name of the condition
        if change_name:
            cond_clean=cond.removeprefix(f"{change_name}_")
        else:
            cond_clean=cond


        
        if select_channels:
            
            picks= dict_select_channels[f"names_channels_{select_channels}"]
            
            epochs_filt = epochs_cond.copy().pick(picks)
        elif select_channels==None:
            epochs_filt=epochs_cond
            
        
        channels = epochs_filt.ch_names
        data = epochs_filt.get_data()   # (n_epochs, n_channels, n_times)
        sfreq = epochs_filt.info['sfreq']
        fmin=epochs_filt.info['highpass']
        fmax=epochs_filt.info['lowpass']

        n_epochs, n_channels, n_times = data.shape

        # Número total de modelos (n_epochs * n_channels)
        shape_tabla = n_epochs * n_channels
        print(f"Number of epochs: {n_epochs}, Number of channels: {n_channels},")

        # Parámetros PSD
        #n_per_seg = el número de muestras por segmento que se usa para calcular una ventana del método de Welch.

        #lower oscillation is the inverse of the lowest frequency, multiplied by 3 to ensure at least 3 cycles are captured
        lower_oscillation=(1/fmin)*3
        n_per_seg = int(lower_oscillation * sfreq)

        psd, freqs = mne.time_frequency.psd_array_welch(
            data,
            sfreq=sfreq,
            fmin=fmin,
            fmax=fmax,
            n_per_seg=n_per_seg,
            # n_fft=n_per_seg,
            n_jobs=20
        )

        # psd: (n_epochs, n_channels, n_freqs)
        psd_flat = psd.reshape(-1, psd.shape[-1])   # (n_epochs * n_channels, n_freqs)
        print("PSD FLAT shape:", psd_flat.shape)

        # Resolución frecuencial (mejor así que con sfreq/n_per_seg)
        freq_resolution = sfreq/n_per_seg
        print("Resolución frecuencial:", freq_resolution, "Hz")

        # Creamos FOOOFGroup para este sujeto
        fg_subject = FOOOFGroup(
            peak_width_limits=[2*freq_resolution, 12],
            max_n_peaks=4,
            min_peak_height=0.2,
            peak_threshold=2.0,
            aperiodic_mode=aperiodic_mode,   # 'fixed' o 'knee'
        )

        # Ajustamos el modelo para TODOS los espectros de este sujeto
        fg_subject.fit(freqs, psd_flat, n_jobs=25)
        print("FOOOF ajustado para el sujeto shaoe:", len(fg_subject))



        band_power_dictionary = {}
        band_ps_multiple=[]
        # Para cada banda definida:
        for label, definition in bands:
            
            # Array para guardar un valor por modelo
            # (power total sumado en esa banda)
            band_power = []
            i=0
            # Recorremos cada modelo FOOOF dentro del FOOOFGroup
            for f_res in fg_subject:
                i+=1
                # Extraemos TODOS los picos dentro de la banda
                band_ps = get_band_peak(
                    f_res.peak_params,
                    definition,
                    select_highest=select_highest
                )
                
                if select_highest==False:
                    # Si hay múltiples picos → sumamos el PW de todos
                    if band_ps.size > 3:
                        # print(f"mas de un pico en fooof {i} para la banda {label}")
                        total_power = np.sum(band_ps[:,1])   # columna 1 = PW
                    else:
                        
                        band_ps=check_nans(band_ps)
                        total_power = band_ps[1]  # columna 1 = PW
                        
                        
                elif select_highest==True:
                    band_ps=check_nans(band_ps)
                    total_power = band_ps[1]  # columna 1 = PW
                    

                band_power.append(total_power)

            # Convertimos a array
            band_power = np.array(band_power)

            # Guardamos en el diccionario
            band_power_dictionary[label] = band_power

        # Mostrar resumen
        for b, vals in band_power_dictionary.items():
            print(f"Banda {b}: {vals.shape} power values (1 valor por modelo)")

        aperiodic = fg_subject.get_params('aperiodic_params')   # array
        if aperiodic_mode=="knee":
            offsets, knees, exponents = aperiodic[:, 0], aperiodic[:, 1], aperiodic[:, 2]
        elif aperiodic_mode=="fixed":
            offsets, exponents = aperiodic[:, 0], aperiodic[:, 1]
            
        # Extraer R² del ajuste FOOOF
        r2 = fg_subject.get_params('r_squared')

        # Extraer error (RMSE del model fit)
        error = fg_subject.get_params('error')

            
            # Crear índices de epochs (0,1,2,... repetidos por canal)
        epoch_idx = np.repeat(np.arange(n_epochs), n_channels)


            
        #append the fooof group to the dictionary
        dict_fg[cond_clean]=fg_subject

        
        # DataFrame del sujeto por condition
        df_fooof_subject_condition = pd.DataFrame({
            'Subject': [subject]* shape_tabla,
            'Condition': [cond_clean]* shape_tabla,
            'Epoch': epoch_idx,
            'Elect': np.tile(channels, n_epochs),
            
            # Band powers
            'delta':  band_power_dictionary.get('delta'),
            'theta':  band_power_dictionary.get('theta'),
            'alpha':  band_power_dictionary.get('alpha'),
            'beta':   band_power_dictionary.get('beta'),
            'gamma':  band_power_dictionary.get('gamma'),
            
            # Aperiodic
            'offsets': offsets,
            'exponents': exponents,
            
            # Metrics
            'r2': r2,
            'error': error
        })

        # Agregar knee en caso de ser necesario
        if aperiodic_mode == "knee":
            df_fooof_subject_condition['knee'] = knees
        list_df.append(df_fooof_subject_condition)
    
    df_fooof_subject = pd.concat(list_df, ignore_index=True)
        
    if return_fg_subject:
        return df_fooof_subject, dict_fg
    else:
        return df_fooof_subject





        

In [7]:
## tests

freq_bands = {
    'delta': (1, 4),
    'theta': (4, 8),
    'alpha': (8, 12),
    'beta': (12, 30),
    'gamma': (30, 40)
}  

bands = Bands(freq_bands)

aperiodic_mode="fixed" # "fixed" or "knee"

# fmin=1
# fmax=40

# select_channels="only_zinnen"

# condition="zinnen"
# condition_epoch=f"fix_{condition.upper()}"
return_fg_subject=True

select_channels=None
filtering=True
if filtering==True:
    lfreq=1
    hfreq=40
    filter_name= f"filt_{lfreq}-{hfreq}"
    print(f"Filtrado aplicado: {lfreq}-{hfreq} Hz")
elif filtering==False:
    print("No se ha aplicado filtrado.")

Filtrado aplicado: 1-40 Hz


In [8]:
selected_subjects = [s for s in subjects if s not in subjects_remove]

In [9]:
df_fooof_subject_all = pd.DataFrame()
fg_subject_all={}

# df_fooof_subject_all_knee = pd.DataFrame()
# fg_subject_all_knee=[]

# for condition in conditions:
    # for h_subj in range(len(subjects)):
# for h_subj in range(1):  # probar con un solo sujeto primero
for subj in selected_subjects:

    path_epochs = epochs_clean_path / f"{subj}_epochs_{layer_script}-epo.fif"

    epochs = mne.read_epochs(path_epochs, preload=True)
    
    if filtering:
        epochs.filter(l_freq=lfreq, h_freq=hfreq, n_jobs=10)
        print(f"Filter applied to {subj}: {lfreq}-{hfreq} Hz")
        filter_applied=True
        
    if layer_script=="block":
        change_name="fix"
    elif layer_script=="event":
        change_name="begin"
    
    # def compute_fooof_subject(epochs,condition, bands,dict_select_channels, aperiodic_mode="fixed", select_highest=False, select_channels=None,return_fg_subject=False ):

    df_fooof_subject, fg_subject= compute_fooof_subject(
        subject=subj,
        epochs=epochs,
        # contidion=condition, im removing it  to include it in the function
        bands=bands,
        dict_select_channels=dict_select_channels,
        aperiodic_mode=aperiodic_mode,
        change_name=change_name,
        select_highest=True,
        select_channels=select_channels,
        return_fg_subject=return_fg_subject
    )
    if return_fg_subject==True:
        fg_subject_all[subj] =fg_subject
        
    # df_fooof_subject_fixed, fg_subject_fixed= compute_fooof_subject(
    #     epochs_condition,
    #     condition,
    #     bands,
    #     dict_select_channels,
    #     aperiodic_mode="fixed",
    #     select_highest=False,
    #     select_channels=select_channels,
    #     return_fg_subject=return_fg_subject
    # )
    # if return_fg_subject==True:
    #     fg_subject_all_fixed.append(fg_subject_fixed)
    

    # Añadir al DataFrame general
    df_fooof_subject_all = pd.concat([df_fooof_subject_all, df_fooof_subject], ignore_index=True)


    del epochs, df_fooof_subject
    if return_fg_subject==True:
        del fg_subject

Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_event\epochs_clean_event\sub-V1001_epochs_event-epo.fif ...
    Found the data of interest:
        t =       0.00 ...    8000.00 ms
        0 CTF compensation matrices available
Not setting metadata
250 matching events found
No baseline correction applied
0 projection items activated
Setting up band-pass filter from 1 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 991 samples (3.303 s)



[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   8 tasks      | elapsed:    1.9s
[Parallel(n_jobs=10)]: Done 116 tasks      | elapsed:    2.0s
[Parallel(n_jobs=10)]: Done 13332 tasks      | elapsed:    4.4s
[Parallel(n_jobs=10)]: Done 45588 tasks      | elapsed:   10.1s
[Parallel(n_jobs=10)]: Done 67300 tasks      | elapsed:   14.2s
[Parallel(n_jobs=10)]: Done 67491 out of 67500 | elapsed:   14.2s remaining:    0.0s
[Parallel(n_jobs=10)]: Done 67500 out of 67500 | elapsed:   14.2s finished


Filter applied to sub-V1001: 1-40 Hz
Number of epochs: 89, Number of channels: 270,
Effective window size : 0.853 (s)


C:\Users\UCM\AppData\Local\Temp\ipykernel_20152\444611167.py:44: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data = epochs_filt.get_data()   # (n_epochs, n_channels, n_times)
[Parallel(n_jobs=20)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done   6 out of  20 | elapsed:    3.2s remaining:    7.5s
[Parallel(n_jobs=20)]: Done  11 out of  20 | elapsed:    3.4s remaining:    2.8s
[Parallel(n_jobs=20)]: Done  16 out of  20 | elapsed:    3.6s remaining:    0.8s
[Parallel(n_jobs=20)]: Done  20 out of  20 | elapsed:    3.8s finished


PSD FLAT shape: (24030, 34)
Resolución frecuencial: 0.3333333333333333 Hz
Running FOOOFGroup across 24030 power spectra.
FOOOF ajustado para el sujeto shaoe: 24030
Banda delta: (24030,) power values (1 valor por modelo)
Banda theta: (24030,) power values (1 valor por modelo)
Banda alpha: (24030,) power values (1 valor por modelo)
Banda beta: (24030,) power values (1 valor por modelo)
Banda gamma: (24030,) power values (1 valor por modelo)
Number of epochs: 15, Number of channels: 270,
Effective window size : 0.853 (s)


...

PSD FLAT shape: (16200, 34)
Resolución frecuencial: 0.3333333333333333 Hz
Running FOOOFGroup across 16200 power spectra.
FOOOF ajustado para el sujeto shaoe: 16200
Banda delta: (16200,) power values (1 valor por modelo)
Banda theta: (16200,) power values (1 valor por modelo)
Banda alpha: (16200,) power values (1 valor por modelo)
Banda beta: (16200,) power values (1 valor por modelo)
Banda gamma: (16200,) power values (1 valor por modelo)
Number of epochs: 8, Number of channels: 270,
Effective window size : 0.853 (s)


C:\Users\UCM\AppData\Local\Temp\ipykernel_20152\444611167.py:44: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data = epochs_filt.get_data()   # (n_epochs, n_channels, n_times)
[Parallel(n_jobs=20)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done   6 out of  20 | elapsed:    0.0s remaining:    0.1s
[Parallel(n_jobs=20)]: Done  11 out of  20 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=20)]: Done  16 out of  20 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=20)]: Done  20 out of  20 | elapsed:    0.0s finished


PSD FLAT shape: (2160, 34)
Resolución frecuencial: 0.3333333333333333 Hz
Running FOOOFGroup across 2160 power spectra.
FOOOF ajustado para el sujeto shaoe: 2160
Banda delta: (2160,) power values (1 valor por modelo)
Banda theta: (2160,) power values (1 valor por modelo)
Banda alpha: (2160,) power values (1 valor por modelo)
Banda beta: (2160,) power values (1 valor por modelo)
Banda gamma: (2160,) power values (1 valor por modelo)
Number of epochs: 4, Number of channels: 270,
Effective window size : 0.853 (s)


C:\Users\UCM\AppData\Local\Temp\ipykernel_20152\444611167.py:44: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data = epochs_filt.get_data()   # (n_epochs, n_channels, n_times)
[Parallel(n_jobs=20)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done   6 out of  20 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=20)]: Done  11 out of  20 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=20)]: Done  16 out of  20 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=20)]: Done  20 out of  20 | elapsed:    0.0s finished


PSD FLAT shape: (1080, 34)
Resolución frecuencial: 0.3333333333333333 Hz
Running FOOOFGroup across 1080 power spectra.
FOOOF ajustado para el sujeto shaoe: 1080
Banda delta: (1080,) power values (1 valor por modelo)
Banda theta: (1080,) power values (1 valor por modelo)
Banda alpha: (1080,) power values (1 valor por modelo)
Banda beta: (1080,) power values (1 valor por modelo)
Banda gamma: (1080,) power values (1 valor por modelo)


In [10]:
# if select_channels:
#     if filtering==True and filter_applied==True:
#         df_fooof_subject_all.to_pickle(ACW_path / f"df_fooof_subject_all_{filter_name}_{aperiodic_mode}_{select_channels}_{layer_script}.pickle")
#         fg_subject_all_path=ACW_path / f"fg_subject_all_{filter_name}_{aperiodic_mode}_{select_channels}_{layer_script}.pkl"
#         print(f"Saved df_fooof_subject_all_{filter_name}_{aperiodic_mode}_{select_channels}_{layer_script}.pickle")
#     else:
#         df_fooof_subject_all.to_pickle(ACW_path / f"df_fooof_subject_all_{aperiodic_mode}_{select_channels}_{layer_script}.pickle")
#         fg_subject_all_path=ACW_path / f"fg_subject_all_{aperiodic_mode}_{select_channels}_{layer_script}.pkl"
#         print(f"Saved df_fooof_subject_all_{aperiodic_mode}_{select_channels}_{layer_script}.pickle")
# else:
#     if filtering==True and filter_applied==True:
#         df_fooof_subject_all.to_pickle(ACW_path / f"df_fooof_subject_all_{filter_name}_{aperiodic_mode}_{layer_script}.pickle")
#         fg_subject_all_path=ACW_path / f"fg_subject_all_{filter_name}_{aperiodic_mode}_{layer_script}.pkl"
#         print(f"Saved df_fooof_subject_all_{filter_name}_{aperiodic_mode}_{layer_script}.pickle")
#     else:   
#         df_fooof_subject_all.to_pickle(ACW_path / f"df_fooof_subject_all_{aperiodic_mode}_{layer_script}.pickle")
#         fg_subject_all_path=ACW_path / f"fg_subject_all_{aperiodic_mode}_{layer_script}.pkl"
#         print(f"Saved df_fooof_subject_all_{aperiodic_mode}_{layer_script}.pickle")

In [11]:

# -------------------------------
# Construcción del sufijo del nombre
# -------------------------------
suffix = []

if filtering and filter_applied:
    suffix.append(filter_name)

suffix.append(aperiodic_mode)

if select_channels:
    suffix.append(select_channels)

suffix.append(layer_script)

suffix_str = "_".join(suffix)

# -------------------------------
# Rutas de guardado
# -------------------------------
df_path = ACW_path / f"df_fooof_subject_all_{suffix_str}.pickle"
fg_subject_all_path = ACW_path / f"fg_subject_all_{suffix_str}.pkl"

# -------------------------------
# Guardar DataFrame
# -------------------------------
df_fooof_subject_all.to_pickle(df_path)
print(f"Saved {df_path.name}")

# -------------------------------
# Guardar FOOOF groups (diccionario anidado)
# -------------------------------
with open(fg_subject_all_path, "wb") as f:
    pickle.dump(fg_subject_all, f)

print(f"Saved {fg_subject_all_path.name}")


Saved df_fooof_subject_all_filt_1-40_fixed_event.pickle
Saved fg_subject_all_filt_1-40_fixed_event.pkl


In [12]:
# import random
# import matplotlib.pyplot as plt
# from fooof import FOOOFGroup

# # =====================================================
# # 1) COMBINAR TODOS LOS FOOOFGroup EN UNO SOLO
# # =====================================================
# def merge_fooofgroups(fg_list):
#     """Une múltiples FOOOFGroup en uno solo."""
    
#     # Crear un FOOOFGroup vacío
#     merged = FOOOFGroup()
    
#     # Recuperar freqs del primero
#     merged.freqs = fg_list[0].freqs
    
#     # Concatenar todos los power spectra
#     merged.power_spectra = np.vstack([fg.power_spectra for fg in fg_list])
    
#     # Combinar los resultados
#     merged.group_results = []
#     for fg in fg_list:
#         merged.group_results.extend(fg.group_results)

#     return merged

# fg_merged = merge_fooofgroups(fg_subject_all)

# import random
# import matplotlib.pyplot as plt

# # 1) Número total de modelos
# n_available = len(fg_merged)
# n_plots = min(50, n_available)

# # 2) Seleccionar 50 modelos aleatorios
# selected_indices = random.sample(range(n_available), n_plots)

# print(f"Guardanddo {n_plots} modelos de {n_available} disponibles.")

# # =============================================================
# #   CREAR CARPETAS
# # =============================================================

# root_path = ACW_path / f"plot_FOOOF_{select_channels}_{aperiodic_mode}_{layer_script}"
# root_path.mkdir(parents=True, exist_ok=True)

# save_path_linear = root_path / f"plot_FOOOF_only_zinnen_{layer_script}_linear_power"
# save_path_linear.mkdir(parents=True, exist_ok=True)

# save_path_log = root_path / f"plot_FOOOF_only_zinnen_{layer_script}_log_power"
# save_path_log.mkdir(parents=True, exist_ok=True)

# # =============================================================
# #   EXTRAER, PLOTEAR Y GUARDAR
# # =============================================================

# for idx in selected_indices:

#     # EXTRAER MODELO INDIVIDUAL
#     fm = fg_merged.get_fooof(idx)

#     # ============================
#     #   PLOT 1: LINEAR
#     # ============================
#     fig_lin, ax_lin = plt.subplots()
#     fm.plot(ax=ax_lin,plot_peaks="shade-dot", plt_log=False)
#     fname_lin = f"fooof_model_{idx:04d}_plot_linear.png"
#     fig_lin.savefig(save_path_linear / fname_lin, dpi=200, bbox_inches="tight")
#     plt.close(fig_lin)

#     # ============================
#     #   PLOT 2: LOG
#     # ============================
#     fig_log, ax_log = plt.subplots()
#     fm.plot(ax=ax_log,plot_peaks="shade-dot", plt_log=True)
#     fname_log = f"fooof_model_{idx:04d}_plot_log.png"
#     fig_log.savefig(save_path_log / fname_log, dpi=200, bbox_inches="tight")
#     plt.close(fig_log)

# print(f"""
# Listo: {n_plots * 2} figuras guardadas en:

# - {save_path_linear}
# - {save_path_log}
# """)


In [13]:
# =====================================================
#   PLOTEAR Y GUARDAR FIGURAS
# =====================================================

# n_available = len(fg_merged)
# n_plots = min(50, n_available)
# selected_indices = random.sample(range(n_available), n_plots)

In [14]:
# import random
# import matplotlib.pyplot as plt
# from fooof import FOOOFGroup

# # =====================================================
# # 1) COMBINAR TODOS LOS FOOOFGroup EN UNO SOLO
# # =====================================================
# def merge_fooofgroups(fg_list):
#     """Une múltiples FOOOFGroup en uno solo."""
    
#     merged = FOOOFGroup()
#     merged.freqs = fg_list[0].freqs
#     merged.power_spectra = np.vstack([fg.power_spectra for fg in fg_list])
    
#     merged.group_results = []
#     for fg in fg_list:
#         merged.group_results.extend(fg.group_results)

#     return merged

# # =====================================================
# #   MERGE DE FG SEGÚN MODO (KNEE O FIXED)
# # =====================================================

# if aperiodic_mode == "fixed":
#     fg_merged = merge_fooofgroups(fg_subject_all_fixed)
# elif aperiodic_mode == "knee":
#     fg_merged = merge_fooofgroups(fg_subject_all_knee)
# else:
#     raise ValueError("aperiodic_mode debe ser 'fixed' o 'knee'")



# print(f"Guardando {n_plots} modelos de {n_available} para modo: {aperiodic_mode}")

#     # =====================================================
# #   CREAR CARPETAS SEGÚN MODO (fixed / knee)
# # =====================================================

# root_path = ACW_path / f"plot_FOOOF_{select_channels}_{layer_script}"
# root_path.mkdir(parents=True, exist_ok=True)

# save_path_linear = root_path / f"plot_FOOOF_{select_channels}_{aperiodic_mode}_{layer_script}_linear_power"
# save_path_linear.mkdir(parents=True, exist_ok=True)

# save_path_log = root_path / f"plot_FOOOF_{select_channels}_{aperiodic_mode}_{layer_script}_log_power"
# save_path_log.mkdir(parents=True, exist_ok=True)

# # =====================================================
# #   LOOP DE PLOTS
# # =====================================================

# for idx in selected_indices:

#     fm = fg_merged.get_fooof(idx)

#     # ----- LINEAR -----
#     fig_lin, ax_lin = plt.subplots()
#     fm.plot(ax=ax_lin, plot_peaks="shade-dot", plt_log=False)
#     fname_lin = f"fooof_model_{idx:04d}_{select_channels}_{aperiodic_mode}_{layer_script}_plot_linear.png"
#     fig_lin.savefig(save_path_linear / fname_lin, dpi=200, bbox_inches="tight")
#     plt.close(fig_lin)

#     # ----- LOG -----
#     fig_log, ax_log = plt.subplots()
#     fm.plot(ax=ax_log, plot_peaks="shade-dot", plt_log=True)
#     fname_log = f"fooof_model_{idx:04d}_{select_channels}_{aperiodic_mode}_{layer_script}_plot_log.png"
#     fig_log.savefig(save_path_log / fname_log, dpi=200, bbox_inches="tight")
#     plt.close(fig_log)

# print(f"""
# Listo: {n_plots * 2} figuras guardadas en:

# - {save_path_linear}
# - {save_path_log}
# """)
